# 2023: Week 3 New Customers
Using SQL to clean and transform banking data.
<br>
The challenge requirements and dataset can be downloaded from [Preppin' Data Challenge](https://preppindata.blogspot.com/2023/01/2023-week-4-new-customers.html)

In [1]:
# Connect SQL to the Notebook
%load_ext sql
%sql duckdb:///week4.duckdb

Connecting to 'duckdb:///week4.duckdb'

Add month number to each table and union tables

In [2]:
%%sql

CREATE OR REPLACE TABLE Customers_N AS

SELECT 
    *,
    1 as Month
FROM read_xlsx('dataset/New_Customers.xlsx', sheet ='January')

UNION ALL

SELECT 
    *,
    2 as Month 
FROM read_xlsx('dataset/New_Customers.xlsx', sheet ='February')

UNION ALL

SELECT 
    *,
    3 as Month
FROM read_xlsx('dataset/New_Customers.xlsx', sheet ='March')

UNION ALL

SELECT 
    *,
    4 as Month
FROM read_xlsx('dataset/New_Customers.xlsx', sheet ='April')

UNION ALL

SELECT 
    *,
    5 as Month
FROM read_xlsx('dataset/New_Customers.xlsx', sheet ='May')

UNION ALL

SELECT 
    *,
    6 as Month
FROM read_xlsx('dataset/New_Customers.xlsx', sheet ='June')

UNION ALL

SELECT
    *,
    7 as Month
FROM read_xlsx('dataset/New_Customers.xlsx', sheet ='July')

UNION ALL

SELECT 
    *,
    8 as Month
FROM read_xlsx('dataset/New_Customers.xlsx', sheet ='August')

UNION ALL

SELECT 
    *,
    9 as Month
FROM read_xlsx('dataset/New_Customers.xlsx', sheet ='September')

UNION ALL

SELECT 
    *,
    10 as Month
FROM read_xlsx('dataset/New_Customers.xlsx', sheet ='October')

UNION ALL

SELECT 
    *,
    11 as Month
FROM read_xlsx('dataset/New_Customers.xlsx', sheet ='November')

UNION ALL

SELECT 
    *,
    12 as Month
FROM read_xlsx('dataset/New_Customers.xlsx', sheet ='December');


Running query in 'duckdb:///week4.duckdb'

Count


Check if data is succesfully unioned

In [ ]:
%%sql

SELECT *
FROM Customers_N;

Running query in 'duckdb:///week4.duckdb'

ID,Joining Day,Demographic,Value,Month
490910,3,Ethnicity,White,1
490910,3,Date of Birth,5/23/1981,1
490910,3,Account Type,Basic,1
369221,18,Ethnicity,Black,1
369221,18,Date of Birth,3/4/2019,1
369221,18,Account Type,Platinum,1
568991,16,Ethnicity,White,1
568991,16,Date of Birth,4/27/1979,1
568991,16,Account Type,Gold,1
191607,9,Ethnicity,White,1


Creating Joining Day colum in dd/mm/yyyy format

In [18]:
%%sql
rollback;
CREATE OR REPLACE TABLE Customers_Ne AS
SELECT
    ID,
    STRFTIME(
        MAKE_DATE(2023, Month, CAST("Joining Day" AS INTEGER)),
        '%d/%m/%Y'
    )   AS Joining_Date,
    Demographic,
    Value
FROM Customers_N;

Running query in 'duckdb:///week4.duckdb'

Count


Check if Joining date is successfully created

In [19]:
%%sql
SELECT *
FROM Customers_Ne;

Running query in 'duckdb:///week4.duckdb'

ID,Joining_Date,Demographic,Value
490910,03/01/2023,Ethnicity,White
490910,03/01/2023,Date of Birth,5/23/1981
490910,03/01/2023,Account Type,Basic
369221,18/01/2023,Ethnicity,Black
369221,18/01/2023,Date of Birth,3/4/2019
369221,18/01/2023,Account Type,Platinum
568991,16/01/2023,Ethnicity,White
568991,16/01/2023,Date of Birth,4/27/1979
568991,16/01/2023,Account Type,Gold
191607,09/01/2023,Ethnicity,White


Pivot the data to create table with one row per customer details

In [ ]:
%%sql
CREATE OR REPLACE TABLE Customers_P AS
PIVOT Customers_Ne
ON Demographic
IN ('Ethnicity', 'Account Type', 'Date of Birth')
USING FIRST(Value)
GROUP BY ID, Joining_Date;

Running query in 'duckdb:///week4.duckdb'

Count


Rearranging columns to create required output

In [ ]:
%%sql
CREATE OR REPLACE TABLE Customer_P AS
SELECT 
    ID,
    Joining_Date,
    "Account Type",
    "Date of Birth",
    Ethnicity
FROM Customers_P;

Running query in 'duckdb:///week4.duckdb'

Count


Check if the columns have been rearranged

In [29]:
%%sql
SELECT *
FROM Customer_P;

Running query in 'duckdb:///week4.duckdb'

ID,Joining_Date,Account Type,Date of Birth,Ethnicity
784685,29/01/2023,Platinum,6/17/1968,White
483606,14/01/2023,Platinum,7/3/1941,Other
554004,27/01/2023,Gold,7/20/1997,Asian
829199,05/01/2023,Basic,1/25/1978,Other
188081,26/02/2023,Basic,5/29/1969,Other
429112,08/02/2023,Basic,11/20/1993,Black
804513,25/02/2023,Gold,5/12/1974,Asian
978249,20/02/2023,Platinum,7/9/1944,Asian
737345,14/02/2023,Gold,1/27/1971,Other
373223,29/03/2023,Basic,11/12/1985,Asian


Check if the table has the correct number of rows

In [30]:
%%sql
SELECT COUNT(*) AS rows
FROM Customers_P;

Running query in 'duckdb:///week4.duckdb'

rows
990
